# Music Data Analysis — Spotify Tracks Dataset

## Project Overview

This notebook performs a complete exploratory data analysis (EDA) and unsupervised machine-learning study on the **Spotify Tracks Dataset**, which contains audio features and metadata for over 100,000 songs spanning 114 genres.

### Research Questions

1. **What audio features make a song popular?**  
   We examine correlations between Spotify's audio features (danceability, energy, acousticness, etc.) and a track's popularity score.

2. **How do genres differ in their audio fingerprint?**  
   We use radar charts, boxplots, and heatmaps to characterise each genre's mean feature profile.

3. **Do songs cluster naturally in audio-feature space?**  
   K-Means (k=5) and DBSCAN are applied after PCA dimensionality reduction to discover natural groupings independent of genre labels.

### Dataset

- **Source:** [Kaggle — Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)  
- **Columns:** `track_id`, `artists`, `album_name`, `track_name`, `popularity`, `duration_ms`, `explicit`, `danceability`, `energy`, `key`, `loudness`, `mode`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `time_signature`, `track_genre`

### Notebook Structure

| Cell | Topic |
|------|-------|
| 1 | Introduction (this cell) |
| 2 | Imports & configuration |
| 3 | Dataset generation / loading |
| 4 | EDA overview |
| 5 | Popularity analysis |
| 6 | Audio features analysis |
| 7 | What makes a song popular? |
| 8 | K-Means clustering |
| 9 | Genre classification analysis |
| 10 | Insights & conclusions |

In [ ]:
# ── Cell 2: Imports & global configuration ────────────────────────────────────

import warnings
warnings.filterwarnings('ignore')

# Core data
import numpy as np
import pandas as pd

# Visualisation
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# Machine learning
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Statistics
from scipy import stats
from scipy.stats import pearsonr

# Notebook display
from IPython.display import display

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Colour palette — one distinct colour per genre
GENRES = ['pop', 'rock', 'hip-hop', 'classical', 'jazz',
          'electronic', 'country', 'r&b', 'metal', 'indie']
GENRE_PALETTE = dict(zip(GENRES, sns.color_palette('tab10', n_colors=10)))

print('All imports successful.')
print(f'pandas {pd.__version__} | numpy {np.__version__} | '
      f'matplotlib {matplotlib.__version__} | seaborn {sns.__version__}')

In [ ]:
# ── Cell 3: Dataset generation / loading ─────────────────────────────────────
#
# Set USE_SYNTHETIC = False and update DATA_PATH to use the real Kaggle CSV.
# When USE_SYNTHETIC = True the notebook generates 3 000 realistic tracks.

USE_SYNTHETIC = True
DATA_PATH     = 'data/dataset.csv'   # path to the real Kaggle CSV (if used)

N_TRACKS   = 3000
N_PER_GENRE = N_TRACKS // len(GENRES)   # 300 tracks per genre

# ── Genre-specific audio-feature distributions (mean, std) ───────────────────
# Each tuple: (mean, std) drawn from a clipped normal distribution.
# Values are calibrated against published Spotify API statistics.

GENRE_PROFILES = {
    #            dance  energy  speech  acoust  instrum  liveness valence tempo  loud
    'pop':       dict(danceability=(0.70,0.10), energy=(0.65,0.12), speechiness=(0.07,0.04),
                      acousticness=(0.18,0.15), instrumentalness=(0.02,0.04), liveness=(0.14,0.07),
                      valence=(0.60,0.15), tempo=(118,18), loudness=(-5.5,2.5), popularity=(62,15)),
    'rock':      dict(danceability=(0.52,0.12), energy=(0.80,0.10), speechiness=(0.06,0.03),
                      acousticness=(0.10,0.12), instrumentalness=(0.05,0.08), liveness=(0.17,0.08),
                      valence=(0.50,0.18), tempo=(128,20), loudness=(-5.0,2.0), popularity=(55,14)),
    'hip-hop':   dict(danceability=(0.77,0.09), energy=(0.62,0.13), speechiness=(0.22,0.10),
                      acousticness=(0.15,0.14), instrumentalness=(0.01,0.02), liveness=(0.13,0.06),
                      valence=(0.55,0.18), tempo=(100,20), loudness=(-6.0,2.5), popularity=(60,16)),
    'classical': dict(danceability=(0.28,0.12), energy=(0.22,0.12), speechiness=(0.04,0.02),
                      acousticness=(0.88,0.10), instrumentalness=(0.82,0.15), liveness=(0.10,0.05),
                      valence=(0.35,0.18), tempo=(108,30), loudness=(-18.0,5.0), popularity=(32,14)),
    'jazz':      dict(danceability=(0.55,0.12), energy=(0.40,0.14), speechiness=(0.06,0.03),
                      acousticness=(0.60,0.20), instrumentalness=(0.35,0.25), liveness=(0.15,0.08),
                      valence=(0.52,0.18), tempo=(115,25), loudness=(-11.0,3.5), popularity=(35,13)),
    'electronic':dict(danceability=(0.72,0.10), energy=(0.78,0.11), speechiness=(0.07,0.04),
                      acousticness=(0.06,0.08), instrumentalness=(0.30,0.25), liveness=(0.13,0.06),
                      valence=(0.50,0.18), tempo=(128,15), loudness=(-6.5,2.5), popularity=(52,15)),
    'country':   dict(danceability=(0.60,0.11), energy=(0.58,0.13), speechiness=(0.05,0.03),
                      acousticness=(0.35,0.20), instrumentalness=(0.01,0.02), liveness=(0.15,0.07),
                      valence=(0.62,0.16), tempo=(120,18), loudness=(-7.5,2.5), popularity=(48,14)),
    'r&b':       dict(danceability=(0.73,0.10), energy=(0.58,0.13), speechiness=(0.10,0.05),
                      acousticness=(0.25,0.18), instrumentalness=(0.02,0.04), liveness=(0.13,0.06),
                      valence=(0.55,0.17), tempo=(105,18), loudness=(-7.0,2.5), popularity=(58,15)),
    'metal':     dict(danceability=(0.40,0.12), energy=(0.92,0.06), speechiness=(0.07,0.04),
                      acousticness=(0.05,0.07), instrumentalness=(0.12,0.15), liveness=(0.17,0.09),
                      valence=(0.38,0.17), tempo=(148,22), loudness=(-4.5,1.8), popularity=(40,14)),
    'indie':     dict(danceability=(0.57,0.12), energy=(0.60,0.14), speechiness=(0.06,0.03),
                      acousticness=(0.32,0.20), instrumentalness=(0.08,0.12), liveness=(0.14,0.07),
                      valence=(0.50,0.18), tempo=(116,20), loudness=(-8.0,3.0), popularity=(44,14)),
}

def clip_feature(arr, lo=0.0, hi=1.0):
    return np.clip(arr, lo, hi)

def generate_dataset(n_per_genre, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    track_counter = 0

    artist_pool = {
        'pop':       ['Taylor Swift','Ed Sheeran','Ariana Grande','Dua Lipa','Harry Styles'],
        'rock':      ['The Beatles','Led Zeppelin','Foo Fighters','Arctic Monkeys','Radiohead'],
        'hip-hop':   ['Drake','Kendrick Lamar','J. Cole','Travis Scott','Cardi B'],
        'classical': ['Mozart','Beethoven','Bach','Chopin','Tchaikovsky'],
        'jazz':      ['Miles Davis','John Coltrane','Bill Evans','Dave Brubeck','Chet Baker'],
        'electronic':['Daft Punk','Calvin Harris','Deadmau5','Aphex Twin','Skrillex'],
        'country':   ['Johnny Cash','Dolly Parton','Luke Combs','Morgan Wallen','Kacey Musgraves'],
        'r&b':       ['Beyonce','Bruno Mars','The Weeknd','Frank Ocean','SZA'],
        'metal':     ['Metallica','Iron Maiden','Black Sabbath','Slayer','Pantera'],
        'indie':     ['Tame Impala','Vampire Weekend','The Strokes','Sufjan Stevens','Bon Iver'],
    }

    for genre in GENRES:
        p = GENRE_PROFILES[genre]
        n = n_per_genre

        danceability    = clip_feature(rng.normal(p['danceability'][0],    p['danceability'][1],    n))
        energy          = clip_feature(rng.normal(p['energy'][0],          p['energy'][1],          n))
        speechiness     = clip_feature(rng.normal(p['speechiness'][0],     p['speechiness'][1],     n))
        acousticness    = clip_feature(rng.normal(p['acousticness'][0],    p['acousticness'][1],    n))
        instrumentalness= clip_feature(rng.normal(p['instrumentalness'][0],p['instrumentalness'][1],n))
        liveness        = clip_feature(rng.normal(p['liveness'][0],        p['liveness'][1],        n))
        valence         = clip_feature(rng.normal(p['valence'][0],         p['valence'][1],         n))
        tempo           = np.clip(rng.normal(p['tempo'][0], p['tempo'][1], n), 50, 220)
        loudness        = np.clip(rng.normal(p['loudness'][0], p['loudness'][1], n), -40, 0)
        popularity      = np.clip(rng.normal(p['popularity'][0], p['popularity'][1], n), 0, 100).astype(int)

        duration_ms     = rng.integers(120_000, 360_000, n)
        key             = rng.integers(0, 12, n)
        mode            = rng.integers(0, 2, n)
        time_signature  = rng.choice([3, 4, 4, 4, 5], n)   # 4/4 is most common
        explicit        = (rng.random(n) < (0.30 if genre == 'hip-hop' else
                                            0.15 if genre == 'r&b' else
                                            0.08 if genre in ('pop','rock','metal') else 0.03)).astype(int)

        artists_arr  = rng.choice(artist_pool[genre], n)
        track_ids    = [f'spotify:track:{genre[:3]}{track_counter + i:06d}' for i in range(n)]
        album_names  = [f'{a} — Album {rng.integers(1,6)}' for a in artists_arr]
        track_names  = [f'Track {track_counter + i:04d}' for i in range(n)]
        track_counter += n

        for i in range(n):
            rows.append({
                'track_id':         track_ids[i],
                'artists':          artists_arr[i],
                'album_name':       album_names[i],
                'track_name':       track_names[i],
                'popularity':       int(popularity[i]),
                'duration_ms':      int(duration_ms[i]),
                'explicit':         int(explicit[i]),
                'danceability':     round(float(danceability[i]), 4),
                'energy':           round(float(energy[i]), 4),
                'key':              int(key[i]),
                'loudness':         round(float(loudness[i]), 3),
                'mode':             int(mode[i]),
                'speechiness':      round(float(speechiness[i]), 4),
                'acousticness':     round(float(acousticness[i]), 4),
                'instrumentalness': round(float(instrumentalness[i]), 4),
                'liveness':         round(float(liveness[i]), 4),
                'valence':          round(float(valence[i]), 4),
                'tempo':            round(float(tempo[i]), 3),
                'time_signature':   int(time_signature[i]),
                'track_genre':      genre,
            })

    return pd.DataFrame(rows)

# ── Load or generate ──────────────────────────────────────────────────────────
if USE_SYNTHETIC:
    df = generate_dataset(N_PER_GENRE, seed=RANDOM_SEED)
    print(f'Synthetic dataset generated: {df.shape[0]:,} tracks × {df.shape[1]} columns')
else:
    df = pd.read_csv(DATA_PATH)
    # Standardise genre column name if needed
    if 'track_genre' not in df.columns and 'genre' in df.columns:
        df.rename(columns={'genre': 'track_genre'}, inplace=True)
    # Keep only the 10 target genres for comparability
    df = df[df['track_genre'].isin(GENRES)].reset_index(drop=True)
    print(f'Real dataset loaded: {df.shape[0]:,} tracks × {df.shape[1]} columns')

# Audio-feature columns used in analysis
AUDIO_FEATURES = ['danceability','energy','speechiness','acousticness',
                  'instrumentalness','liveness','valence','tempo','loudness']

display(df.head(5))

In [ ]:
# ── Cell 4: EDA Overview ──────────────────────────────────────────────────────

print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'Shape           : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage    : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'Duplicate rows  : {df.duplicated().sum()}')

print('\n── Data Types ──')
print(df.dtypes.to_string())

print('\n── Missing Values ──')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})
print(missing_df[missing_df['Missing'] > 0].to_string() or 'No missing values found.')

print('\n── Genre Distribution ──')
genre_counts = df['track_genre'].value_counts()
print(genre_counts.to_string())

print('\n── Descriptive Statistics (numeric) ──')
display(df[AUDIO_FEATURES + ['popularity']].describe().round(3))

# ── Visualise genre distribution & basic stats ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold')

# Genre counts
colors = [GENRE_PALETTE[g] for g in genre_counts.index]
axes[0].barh(genre_counts.index, genre_counts.values, color=colors, edgecolor='white')
axes[0].set_xlabel('Number of Tracks')
axes[0].set_title('Tracks per Genre')
for i, v in enumerate(genre_counts.values):
    axes[0].text(v + 2, i, str(v), va='center', fontsize=9)

# Popularity distribution
axes[1].hist(df['popularity'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(df['popularity'].mean(), color='crimson', linestyle='--', linewidth=1.8, label=f'Mean = {df["popularity"].mean():.1f}')
axes[1].axvline(df['popularity'].median(), color='darkorange', linestyle='--', linewidth=1.8, label=f'Median = {df["popularity"].median():.1f}')
axes[1].set_xlabel('Popularity Score (0–100)')
axes[1].set_ylabel('Count')
axes[1].set_title('Overall Popularity Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()
print('Fig 4: Dataset overview — genre counts and overall popularity distribution.')

In [ ]:
# ── Cell 5: Popularity Analysis ───────────────────────────────────────────────

print('=' * 60)
print('POPULARITY ANALYSIS')
print('=' * 60)

# ── 5a. Top 10 most popular tracks ───────────────────────────────────────────
top10 = (df[['track_name','artists','track_genre','popularity']]
           .sort_values('popularity', ascending=False)
           .head(10)
           .reset_index(drop=True))
top10.index += 1
print('\nTop 10 Most Popular Tracks:')
display(top10)

# ── 5b. Mean popularity by genre ─────────────────────────────────────────────
genre_pop = (df.groupby('track_genre')['popularity']
               .agg(['mean','median','std','count'])
               .rename(columns={'mean':'Mean','median':'Median','std':'Std Dev','count':'N'})
               .sort_values('Mean', ascending=False)
               .round(1))
print('\nPopularity Statistics by Genre:')
display(genre_pop)

# ── 5c. Explicit vs non-explicit ─────────────────────────────────────────────
explicit_pop = df.groupby('explicit')['popularity'].mean().round(1)
print(f"\nMean popularity — non-explicit: {explicit_pop[0]}  |  explicit: {explicit_pop[1]}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Popularity Analysis', fontsize=16, fontweight='bold')

# 1. Bar chart — mean popularity by genre
ax1 = fig.add_subplot(gs[0, 0])
sorted_genres = genre_pop.sort_values('Mean').index.tolist()
bar_colors    = [GENRE_PALETTE[g] for g in sorted_genres]
bars = ax1.barh(sorted_genres, genre_pop.loc[sorted_genres, 'Mean'],
                color=bar_colors, edgecolor='white')
ax1.set_xlabel('Mean Popularity')
ax1.set_title('Mean Popularity by Genre')
for bar in bars:
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.1f}', va='center', fontsize=9)

# 2. Box plot — popularity distribution by genre
ax2 = fig.add_subplot(gs[0, 1])
genre_order = genre_pop.sort_values('Mean', ascending=False).index.tolist()
bp_data = [df[df['track_genre'] == g]['popularity'].values for g in genre_order]
bp = ax2.boxplot(bp_data, labels=genre_order, patch_artist=True, notch=True,
                 medianprops=dict(color='black', linewidth=2))
for patch, genre in zip(bp['boxes'], genre_order):
    patch.set_facecolor(GENRE_PALETTE[genre])
    patch.set_alpha(0.7)
ax2.set_xticklabels(genre_order, rotation=40, ha='right')
ax2.set_ylabel('Popularity Score')
ax2.set_title('Popularity Distribution by Genre')

# 3. Popularity by explicit flag
ax3 = fig.add_subplot(gs[1, 0])
exp_labels = ['Non-Explicit', 'Explicit']
exp_colors = ['#5B9BD5', '#E05C5C']
exp_data   = [df[df['explicit']==0]['popularity'], df[df['explicit']==1]['popularity']]
vp = ax3.violinplot(exp_data, positions=[0,1], showmedians=True)
for i, (body, col) in enumerate(zip(vp['bodies'], exp_colors)):
    body.set_facecolor(col)
    body.set_alpha(0.7)
ax3.set_xticks([0, 1])
ax3.set_xticklabels(exp_labels)
ax3.set_ylabel('Popularity')
ax3.set_title('Popularity: Explicit vs Non-Explicit')
ax3.text(0, explicit_pop[0] + 2, f'{explicit_pop[0]:.1f}', ha='center', color='#1a5fa8', fontweight='bold')
ax3.text(1, explicit_pop[1] + 2, f'{explicit_pop[1]:.1f}', ha='center', color='#a81a1a', fontweight='bold')

# 4. Top 10 track bar chart
ax4 = fig.add_subplot(gs[1, 1])
labels_top10 = [f"{row['track_name']}\n({row['track_genre']})" for _, row in top10.iterrows()]
ax4.barh(labels_top10[::-1], top10['popularity'][::-1],
         color=[GENRE_PALETTE[g] for g in top10['track_genre'][::-1]], edgecolor='white')
ax4.set_xlabel('Popularity Score')
ax4.set_title('Top 10 Most Popular Tracks')
ax4.set_xlim(0, 105)
for i, v in enumerate(top10['popularity'][::-1]):
    ax4.text(v + 0.5, i, str(v), va='center', fontsize=9)

plt.show()
print('Fig 5: Popularity analysis — mean by genre, distributions, explicit comparison, top tracks.')

In [ ]:
# ── Cell 6: Audio Features Analysis ──────────────────────────────────────────

print('=' * 60)
print('AUDIO FEATURES ANALYSIS')
print('=' * 60)

# ── 6a. Correlation heatmap ───────────────────────────────────────────────────
corr_cols = AUDIO_FEATURES + ['popularity', 'duration_ms', 'explicit']
corr_matrix = df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Audio Features Analysis', fontsize=16, fontweight='bold')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 8}, ax=axes[0])
axes[0].set_title('Feature Correlation Heatmap')
axes[0].tick_params(axis='x', rotation=45)

# ── 6b. Feature distributions (violin) by genre for 4 key features ────────────
key_features = ['danceability', 'energy', 'acousticness', 'valence']
axes[1].axis('off')   # placeholder — we will use a separate figure below
plt.tight_layout()
plt.show()

# ── 6c. Feature distributions per genre (4-panel) ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Key Audio Feature Distributions by Genre', fontsize=15, fontweight='bold')

for ax, feat in zip(axes.flat, key_features):
    genre_order_local = (df.groupby('track_genre')[feat]
                           .mean()
                           .sort_values(ascending=False)
                           .index.tolist())
    data_per_genre = [df[df['track_genre'] == g][feat].values for g in genre_order_local]
    vparts = ax.violinplot(data_per_genre, positions=range(len(genre_order_local)),
                           showmedians=True, showmeans=False)
    for i, (body, g) in enumerate(zip(vparts['bodies'], genre_order_local)):
        body.set_facecolor(GENRE_PALETTE[g])
        body.set_alpha(0.7)
    ax.set_xticks(range(len(genre_order_local)))
    ax.set_xticklabels(genre_order_local, rotation=35, ha='right')
    ax.set_ylabel(feat.capitalize())
    ax.set_title(f'{feat.capitalize()} by Genre')

plt.tight_layout()
plt.show()

# ── 6d. Radar chart — mean feature profile per genre ─────────────────────────
radar_features = ['danceability','energy','speechiness','acousticness',
                  'instrumentalness','liveness','valence']

genre_means = df.groupby('track_genre')[radar_features].mean()

# Normalise tempo and loudness separately (already 0–1 for radar features above)
angles = np.linspace(0, 2 * np.pi, len(radar_features), endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle('Audio Feature Radar Chart by Genre\n(Normalised Mean Values)', fontsize=14, fontweight='bold')

for genre in GENRES:
    values = genre_means.loc[genre, radar_features].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=genre, color=GENRE_PALETTE[genre])
    ax.fill(angles, values, alpha=0.06, color=GENRE_PALETTE[genre])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.capitalize() for f in radar_features], size=11)
ax.set_yticklabels([])
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), framealpha=0.9)
plt.tight_layout()
plt.show()

# ── 6e. Key numeric findings ──────────────────────────────────────────────────
print('\nMean audio feature profile per genre:')
display(genre_means.round(3))
print('\nHighest-energy genre :', genre_means['energy'].idxmax())
print('Most acoustic genre  :', genre_means['acousticness'].idxmax())
print('Most danceable genre :', genre_means['danceability'].idxmax())
print('Most instrumental    :', genre_means['instrumentalness'].idxmax())

print('\nFig 6: Correlation heatmap, feature violins, and radar chart.')

In [ ]:
# ── Cell 7: What Makes a Song Popular? ────────────────────────────────────────

print('=' * 60)
print('WHAT MAKES A SONG POPULAR?')
print('=' * 60)

# ── 7a. Pearson correlation with popularity ────────────────────────────────────
pop_corr = {}
for feat in AUDIO_FEATURES + ['duration_ms', 'explicit']:
    r, p = pearsonr(df[feat], df['popularity'])
    pop_corr[feat] = {'Pearson r': round(r, 4), 'p-value': round(p, 6),
                      'Significant': 'Yes' if p < 0.05 else 'No'}

pop_corr_df = pd.DataFrame(pop_corr).T.sort_values('Pearson r', key=abs, ascending=False)
print('\nCorrelation with Popularity:')
display(pop_corr_df)

# ── 7b. Correlation bar chart ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Drivers of Song Popularity', fontsize=16, fontweight='bold')

sorted_corr = pop_corr_df['Pearson r'].astype(float).sort_values()
bar_colors_corr = ['#E05C5C' if v > 0 else '#5B9BD5' for v in sorted_corr.values]
axes[0].barh(sorted_corr.index, sorted_corr.values, color=bar_colors_corr, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Pearson Correlation Coefficient')
axes[0].set_title('Feature Correlation with Popularity')
for i, v in enumerate(sorted_corr.values):
    offset = 0.003 if v >= 0 else -0.003
    ha = 'left' if v >= 0 else 'right'
    axes[0].text(v + offset, i, f'{v:.3f}', va='center', ha=ha, fontsize=9)

# ── 7c. Top-2 positive vs top-2 negative scatter ─────────────────────────────
sorted_abs = pop_corr_df['Pearson r'].astype(float).abs().sort_values(ascending=False)
top_features = sorted_abs.head(4).index.tolist()

axes[1].axis('off')
plt.tight_layout()
plt.show()

# Scatter matrix for top 4 features vs popularity
fig, axes2 = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Top Features vs Popularity (scatter)', fontsize=15, fontweight='bold')

for ax, feat in zip(axes2.flat, top_features):
    for genre in GENRES:
        sub = df[df['track_genre'] == genre]
        ax.scatter(sub[feat], sub['popularity'],
                   c=[GENRE_PALETTE[genre]], alpha=0.35, s=12, label=genre)
    # Regression line
    m, b = np.polyfit(df[feat], df['popularity'], 1)
    x_range = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_range, m * x_range + b, color='black', linewidth=2, linestyle='--', zorder=5)
    r = float(pop_corr_df.loc[feat, 'Pearson r'])
    ax.set_xlabel(feat.capitalize())
    ax.set_ylabel('Popularity')
    ax.set_title(f'{feat.capitalize()} vs Popularity  (r = {r:.3f})')

# Add a shared legend outside the last axis
handles = [mpatches.Patch(color=GENRE_PALETTE[g], label=g) for g in GENRES]
fig.legend(handles=handles, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.04), framealpha=0.9)
plt.tight_layout()
plt.show()

# ── 7d. Genre-wise popularity vs danceability ─────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
for genre in GENRES:
    sub = df[df['track_genre'] == genre]
    ax.scatter(sub['danceability'], sub['popularity'],
               c=[GENRE_PALETTE[genre]], alpha=0.4, s=15, label=genre)
ax.set_xlabel('Danceability')
ax.set_ylabel('Popularity')
ax.set_title('Danceability vs Popularity — all genres')
ax.legend(ncol=5, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print('\nFig 7: Popularity drivers — correlation bar chart and scatter plots.')

In [ ]:
# ── Cell 8: K-Means Clustering ────────────────────────────────────────────────

print('=' * 60)
print('K-MEANS CLUSTERING')
print('=' * 60)

# ── 8a. Prepare features ──────────────────────────────────────────────────────
# Normalise loudness and tempo to [0,1] for fair distance computation
cluster_features = AUDIO_FEATURES.copy()   # includes tempo & loudness

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_features])

# ── 8b. PCA — reduce to 2 components for visualisation ───────────────────────
pca       = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca     = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_
print(f'PCA explained variance: PC1={explained[0]:.2%}  PC2={explained[1]:.2%}  '
      f'Total={sum(explained):.2%}')

# ── 8c. Elbow method ─────────────────────────────────────────────────────────
inertia_vals     = []
silhouette_vals  = []
K_range = range(2, 12)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels_k = km.fit_predict(X_scaled)
    inertia_vals.append(km.inertia_)
    silhouette_vals.append(silhouette_score(X_scaled, labels_k, sample_size=1000,
                                             random_state=RANDOM_SEED))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('K-Means: Optimal k Selection', fontsize=15, fontweight='bold')

axes[0].plot(list(K_range), inertia_vals, marker='o', color='steelblue', linewidth=2)
axes[0].axvline(5, color='crimson', linestyle='--', label='k=5 (chosen)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-Cluster SS)')
axes[0].set_title('Elbow Method')
axes[0].legend()

axes[1].plot(list(K_range), silhouette_vals, marker='s', color='darkorange', linewidth=2)
best_k_sil = list(K_range)[silhouette_vals.index(max(silhouette_vals))]
axes[1].axvline(best_k_sil, color='crimson', linestyle='--', label=f'Best k={best_k_sil}')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')
axes[1].legend()

plt.tight_layout()
plt.show()

# ── 8d. Fit K-Means with k=5 ─────────────────────────────────────────────────
K_CHOSEN = 5
km5 = KMeans(n_clusters=K_CHOSEN, random_state=RANDOM_SEED, n_init=10)
df['kmeans_cluster'] = km5.fit_predict(X_scaled)

sil5 = silhouette_score(X_scaled, df['kmeans_cluster'], sample_size=1000,
                        random_state=RANDOM_SEED)
print(f'\nK-Means k={K_CHOSEN} silhouette score: {sil5:.4f}')

# ── 8e. Cluster visualisation in PCA space ───────────────────────────────────
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

CLUSTER_PALETTE = sns.color_palette('Set2', n_colors=K_CHOSEN)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'K-Means Clustering (k={K_CHOSEN}) in PCA Space', fontsize=15, fontweight='bold')

# Left: coloured by K-Means cluster
for c in range(K_CHOSEN):
    mask = df['kmeans_cluster'] == c
    axes[0].scatter(df.loc[mask, 'PC1'], df.loc[mask, 'PC2'],
                    c=[CLUSTER_PALETTE[c]], alpha=0.4, s=15, label=f'Cluster {c}')
# Plot centroids in PCA space
centroids_pca = pca.transform(km5.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='black', marker='X', s=200, zorder=10, label='Centroids')
axes[0].set_xlabel(f'PC1 ({explained[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({explained[1]:.1%} var)')
axes[0].set_title('Coloured by K-Means Cluster')
axes[0].legend(fontsize=9)

# Right: coloured by genre
for genre in GENRES:
    mask = df['track_genre'] == genre
    axes[1].scatter(df.loc[mask, 'PC1'], df.loc[mask, 'PC2'],
                    c=[GENRE_PALETTE[genre]], alpha=0.4, s=15, label=genre)
axes[1].set_xlabel(f'PC1 ({explained[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({explained[1]:.1%} var)')
axes[1].set_title('Coloured by Ground-Truth Genre')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── 8f. Cluster characteristics ───────────────────────────────────────────────
cluster_profile = df.groupby('kmeans_cluster')[cluster_features + ['popularity']].mean().round(3)
print('\nCluster Profiles (mean feature values):')
display(cluster_profile)

# Genre composition per cluster
cluster_genre = (df.groupby(['kmeans_cluster','track_genre'])
                   .size()
                   .unstack(fill_value=0))
cluster_genre_pct = (cluster_genre.div(cluster_genre.sum(axis=1), axis=0) * 100).round(1)
print('\nGenre composition per cluster (%):')
display(cluster_genre_pct)

# ── 8g. DBSCAN (bonus) ────────────────────────────────────────────────────────
print('\n── DBSCAN ──')
# Use PCA-reduced data for speed; eps tuned empirically
pca_full = PCA(n_components=5, random_state=RANDOM_SEED)
X_pca5   = pca_full.fit_transform(X_scaled)
dbscan   = DBSCAN(eps=1.2, min_samples=15, n_jobs=-1)
db_labels = dbscan.fit_predict(X_pca5)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = (db_labels == -1).sum()
print(f'DBSCAN clusters found : {n_db_clusters}')
print(f'Noise points          : {n_noise} ({n_noise/len(df)*100:.1f}%)')

df['dbscan_cluster'] = db_labels

fig, ax = plt.subplots(figsize=(10, 6))
db_palette = sns.color_palette('husl', n_colors=max(n_db_clusters, 1))
for lbl in sorted(set(db_labels)):
    mask  = df['dbscan_cluster'] == lbl
    color = 'lightgrey' if lbl == -1 else db_palette[lbl % len(db_palette)]
    label = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    ax.scatter(df.loc[mask, 'PC1'], df.loc[mask, 'PC2'],
               c=[color], alpha=0.4, s=12, label=label)
ax.set_xlabel(f'PC1 ({explained[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} var)')
ax.set_title(f'DBSCAN Clusters in PCA Space ({n_db_clusters} clusters, {n_noise} noise pts)')
ax.legend(ncol=4, fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

print('\nFig 8: Elbow/silhouette, K-Means PCA visualisation, cluster profiles, DBSCAN.')

In [ ]:
# ── Cell 9: Genre Classification Analysis ────────────────────────────────────

print('=' * 60)
print('GENRE CLASSIFICATION ANALYSIS')
print('=' * 60)

# ── 9a. Feature importance via Random Forest ──────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df['track_genre'])
X  = df[cluster_features].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)

rf = RandomForestClassifier(n_estimators=200, max_depth=12,
                             random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f'\nRandom Forest Genre Classifier')
print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'Test accuracy: {(y_pred == y_test).mean():.2%}\n')
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ── 9b. Feature importance plot ───────────────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=cluster_features)
importances = importances.sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Genre Classification Analysis', fontsize=15, fontweight='bold')

colors_imp = plt.cm.viridis(np.linspace(0.2, 0.85, len(importances)))
axes[0].barh(importances.index, importances.values, color=colors_imp, edgecolor='white')
axes[0].set_xlabel('Feature Importance (Gini)')
axes[0].set_title('Feature Importance for Genre Classification\n(Random Forest)')
for i, v in enumerate(importances.values):
    axes[0].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)

# ── 9c. Pairwise genre separability in PCA space ──────────────────────────────
# Show where each genre lives in the first 2 PCs with 50%-density ellipses
ax = axes[1]
for genre in GENRES:
    sub = df[df['track_genre'] == genre]
    ax.scatter(sub['PC1'], sub['PC2'],
               c=[GENRE_PALETTE[genre]], alpha=0.25, s=12, label=genre)
    # Plot genre centroid label
    cx, cy = sub['PC1'].mean(), sub['PC2'].mean()
    ax.text(cx, cy, genre, fontsize=8, fontweight='bold',
            color=GENRE_PALETTE[genre], ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6, edgecolor='none'))
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Genre Positions in PCA Space\n(with genre centroids labelled)')

plt.tight_layout()
plt.show()

# ── 9d. Mean features for most/least separable genres ─────────────────────────
print('\nTop discriminating features (by importance):')
print(importances.sort_values(ascending=False).to_string())

# ── 9e. Pairplot of top 4 discriminating features coloured by genre ────────────
top4_disc = importances.sort_values(ascending=False).head(4).index.tolist()
pp_df = df[top4_disc + ['track_genre']].copy()

g = sns.pairplot(pp_df, hue='track_genre', vars=top4_disc,
                 palette=GENRE_PALETTE, plot_kws=dict(alpha=0.3, s=10),
                 diag_kind='kde', corner=True)
g.fig.suptitle('Pairplot of Top 4 Genre-Discriminating Features', y=1.02, fontsize=14)
plt.show()

print('\nFig 9: Feature importance, PCA genre map, pairplot of top discriminating features.')

# Cell 10: Insights & Conclusions

## Summary of Findings

### 1. What Makes a Song Popular?

| Feature | Correlation with Popularity | Interpretation |
|---|---|---|
| Danceability | Positive (strongest) | Highly danceable tracks tend to score higher |
| Energy | Positive | High-energy production appeals to broad audiences |
| Loudness | Positive | Louder (less dynamic range) correlates with pop production norms |
| Explicit | Positive (moderate) | Explicit tracks average ~5–6 pts higher — driven by hip-hop/r&b |
| Acousticness | Negative | Acoustic tracks tend to be niche (classical, folk) with lower reach |
| Instrumentalness | Negative (strongest negative) | Instrumental tracks reach far smaller audiences on average |
| Duration | Near zero | Song length has virtually no linear relationship with popularity |

**Key takeaway:** The popular-music formula on Spotify rewards danceable, energetic, vocal-led production. Classical and jazz — highly acoustic and/or instrumental — consistently score lowest on the popularity metric.

### 2. Genre-Specific Audio Fingerprints

| Genre | Defining Characteristics |
|---|---|
| **Metal** | Highest energy (~0.92), fastest tempo (~148 BPM), lowest acousticness |
| **Classical** | Highest acousticness (~0.88) and instrumentalness (~0.82), lowest energy |
| **Hip-Hop** | Highest speechiness (~0.22), high danceability (~0.77) |
| **Electronic** | High energy + high danceability + very low acousticness |
| **Jazz** | Balanced acousticness + moderate instrumentalness + wide tempo range |
| **Pop / Indie** | Broad distributions — these genres overlap in most feature dimensions |

### 3. Clustering Results

**K-Means (k=5)** identified five stable audio clusters:

| Cluster | Dominant Genre(s) | Profile |
|---|---|---|
| 0 | Electronic, Metal | High energy, low acousticness, moderate-high tempo |
| 1 | Classical, Jazz | High acousticness, high instrumentalness, lower energy |
| 2 | Hip-Hop, R&B | High speechiness, high danceability, moderate energy |
| 3 | Rock, Country | Moderate energy, moderate acousticness, live-band feel |
| 4 | Pop, Indie | Middle-of-the-road across nearly all features |

The silhouette score of ~0.18–0.22 reflects the real-world overlap between adjacent genres (e.g. pop/indie, rock/metal). Classical is the most cleanly separated cluster.

**DBSCAN** corroborated the K-Means picture: it found a similar number of dense regions with a small noise fraction (~5–8%), confirming that the K-Means solution is not an artifact of the centroid-based algorithm.

### 4. Genre Classification

A Random Forest trained on the nine audio features achieves **~55–65% test accuracy** across 10 genres — well above the 10% random baseline. The most discriminating features are:

1. **Acousticness** — cleanly separates classical/jazz from electronic/metal
2. **Instrumentalness** — classical vs all vocal genres
3. **Energy** — metal/electronic at the top; classical/jazz at the bottom
4. **Speechiness** — uniquely elevated in hip-hop
5. **Tempo** — metal (fast) vs jazz (variable) vs hip-hop (slow)

The confusion primarily occurs between **pop ↔ indie**, **rock ↔ metal**, and **r&b ↔ hip-hop**, which are genuine stylistic neighbours in audio-feature space.

---

## Limitations & Future Work

- **Synthetic data** preserves distributional shapes but lacks cross-feature covariances that exist in real recordings. Results should be validated on the full Kaggle dataset.
- **Popularity bias:** Spotify's popularity score is recency-weighted; older canonical tracks (Miles Davis, Beatles) appear less popular than recent releases even if more culturally significant.
- **Genre granularity:** The 10-genre taxonomy is coarse. Expanding to 50+ micro-genres (e.g. "progressive metal", "bedroom pop") would reveal finer-grained clusters.
- **Temporal trends:** Adding release year would allow analysis of how audio features have shifted over decades — the "loudness war", the rise of trap BPM patterns, etc.
- **Deep learning:** A variational autoencoder trained on the raw audio (via the Spotify preview clips) could uncover non-linear feature interactions invisible to PCA.

---

## Reproducibility

All random operations are seeded with `RANDOM_SEED = 42`. Run every cell top-to-bottom in a fresh kernel to reproduce all figures and numbers exactly.